In [ ]:
import hashlib, json, time
from collections import Counter, defaultdict
from pathlib import Path

root=Path('/kaggle/input/datasets/tunguz/xview2-challenge-dataset-train-and-test/train/train')
images=root/'images'; labels=root/'labels'
if not images.is_dir() or not labels.is_dir(): raise RuntimeError('Mounted xBD training images or labels missing')
pre_paths=sorted(images.glob('*_pre_disaster.png'))
if len(pre_paths)!=2799: raise RuntimeError(f'Expected 2799 pre-event images, found {len(pre_paths)}')
records=[]
for pre in pre_paths:
    sid=pre.name[:-len('_pre_disaster.png')]
    event, separator, tail=sid.rpartition('_')
    if not separator or not tail.isdigit(): raise RuntimeError(f'Invalid event identity: {sid}')
    post=images/(sid+'_post_disaster.png'); label=labels/(sid+'_post_disaster.json'); pre_label=labels/(sid+'_pre_disaster.json')
    if not post.is_file() or not label.is_file() or not pre_label.is_file(): raise RuntimeError(f'Missing pair or label: {sid}')
    records.append({'sample_id':sid,'event':event,'pre':pre,'post':post,'label':label,'pre_label':pre_label})
events=sorted({r['event'] for r in records})
val_events={event for i,event in enumerate(events) if i%5==0}
train_events=set(events)-val_events
if not train_events or not val_events or train_events & val_events: raise RuntimeError('Invalid grouped event split')
counts=Counter(); hashes=defaultdict(set); rows=[]; start=time.monotonic()
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): digest.update(block)
    return digest.hexdigest()
for index,r in enumerate(records,1):
    split='validation' if r['event'] in val_events else 'train'
    row={'sample_id':r['sample_id'],'event':r['event'],'split':split}
    for key in ('pre','post','label','pre_label'):
        row[key+'_relative_path']=str(r[key].relative_to(root))
        row[key+'_sha256']=sha256(r[key])
        if key in ('pre','post'): hashes[row[key+'_sha256']].add(split)
    rows.append(row); counts[split]+=1
    if index%100==0: print(f'Hashed {index}/{len(records)} pairs; {time.monotonic()-start:.1f}s',flush=True)
if counts['train']!=2660 or counts['validation']!=139: raise RuntimeError(f'Unexpected split counts: {dict(counts)}')
cross=[digest for digest,splits in hashes.items() if len(splits)>1]
out=Path('/kaggle/working/rv4_xbd_grouped_manifest.jsonl')
with out.open('w',encoding='utf-8') as f:
    for row in rows: f.write(json.dumps(row,sort_keys=True,separators=(',',':'))+'\n')
report={'status':'FAIL' if cross else 'PASS','dataset':'tunguz/xview2-challenge-dataset-train-and-test','scope':'mounted train pair labels only; cross-dataset overlap not checked','split_algorithm':'sorted event names, every fifth event validation, seed-independent','counts':dict(counts),'events_train':sorted(train_events),'events_validation':sorted(val_events),'cross_split_image_hashes':cross,'manifest_sha256':sha256(out),'elapsed_seconds':round(time.monotonic()-start,2)}
Path('/kaggle/working/rv4_xbd_grouped_manifest_report.json').write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
print(json.dumps(report,indent=2,sort_keys=True),flush=True)
